# Tier 1 Performance Benchmark — Pipeline Optimization

**Purpose:** Measure pipeline performance after Tier 1 optimizations and compare against baseline.

## Baseline (pre-Tier 1)

| Difficulty | Time (s) | Iterations |
|---|---|---|
| recall | 125.1 | 1 |
| comprehension | 122.7 | 1 |
| application | 539.9 | 2 |
| high_application | 1290.8 | 4 |
| **TOTAL** | **2078.5** | — |

## Tier 1 Optimizations Applied

- **T-OPT-1.1:** Parallel parse batches via `asyncio.gather()` with `Semaphore(3)`
- **T-OPT-1.2:** Cache Vertex AI Search context on retry iterations
- **T-OPT-1.3:** Skip supervisor on retry (reviewer fail → math_agent directly)

## Targets

- `high_application` ≤ 800s (was 1290.8s)
- Total pipeline ≤ 1400s (was 2078.5s)
- 100% delivery maintained (40/40 items)

## Section 1: Setup & Imports

In [1]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
os.environ.setdefault('ENV', 'develop')

from src.api.schemas import (
    GenerationRequest, GameType, DifficultyLevel, DocScope, GameContentResponse
)
from src.graph.builder import compile_graph

print('Imports OK')

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## Section 2: E2E Runner

In [2]:
NUM_QUESTIONS = 10
TOPIC = "Đạo hàm và ứng dụng"

# Baseline numbers (pre-Tier 1)
BASELINE = {
    "recall": {"time": 125.1, "iterations": 1},
    "comprehension": {"time": 122.7, "iterations": 1},
    "application": {"time": 539.9, "iterations": 2},
    "high_application": {"time": 1290.8, "iterations": 4},
}
BASELINE_TOTAL = sum(b["time"] for b in BASELINE.values())


async def run_e2e(difficulty: str):
    """Run pipeline end-to-end and return (num_delivered, elapsed, iterations, errors)."""
    req = GenerationRequest(
        user_id="perf_benchmark_tier1",
        topic=TOPIC,
        game_types=[GameType.QUIZ],
        num_questions=NUM_QUESTIONS,
        difficulty=DifficultyLevel(difficulty),
        doc_scope=DocScope.SYSTEM,
    )
    app = compile_graph()
    initial_state = {
        "request": req,
        "doc_scope": req.doc_scope.value,
        "iteration_count": 0,
        "rejected_items": [],
        "reviewed_items": [],
        "errors": [],
    }
    t0 = time.time()
    result = await app.ainvoke(initial_state)
    elapsed = time.time() - t0

    output: GameContentResponse | None = result.get("final_output")
    delivered = len(output.content.quiz) if output else 0
    iterations = result.get("iteration_count", 0)
    errors = result.get("errors", [])
    return delivered, elapsed, iterations, errors


print(f"Runner ready — Topic: {TOPIC} | {NUM_QUESTIONS} questions per level")

Runner ready — Topic: Đạo hàm và ứng dụng | 10 questions per level


## Section 3: Run Benchmark (All 4 Difficulty Levels)

In [3]:
print("=" * 70)
print("TIER 1 PERFORMANCE BENCHMARK")
print(f"Topic: {TOPIC} | Requested: {NUM_QUESTIONS} quiz per level")
print("=" * 70)

e2e_results = {}
for diff in ["recall", "comprehension", "application", "high_application"]:
    print(f"\n--- Running {diff} ---")
    delivered, elapsed, iters, errors = await run_e2e(diff)
    pct = delivered / NUM_QUESTIONS * 100
    status = "✅" if delivered >= NUM_QUESTIONS else "❌"
    baseline_t = BASELINE[diff]["time"]
    speedup = (baseline_t - elapsed) / baseline_t * 100
    print(f"  {status} Delivered: {delivered}/{NUM_QUESTIONS} ({pct:.0f}%) in {elapsed:.1f}s ({iters} iter)")
    print(f"  ⏱️  Baseline: {baseline_t:.1f}s → Now: {elapsed:.1f}s ({speedup:+.1f}%)")
    if errors:
        print(f"  ⚠️ Errors: {errors}")
    e2e_results[diff] = {
        "delivered": delivered,
        "requested": NUM_QUESTIONS,
        "pct": pct,
        "elapsed": elapsed,
        "iterations": iters,
        "passed": delivered >= NUM_QUESTIONS,
        "baseline": baseline_t,
        "speedup_pct": speedup,
    }

TIER 1 PERFORMANCE BENCHMARK
Topic: Đạo hàm và ứng dụng | Requested: 10 quiz per level

--- Running recall ---
2026-04-09 13:55:41 [info     ] creating_graph                
2026-04-09 13:55:41 [info     ] graph_created                 
2026-04-09 13:55:41 [info     ] graph_compiled_without_checkpointer
2026-04-09 13:55:41 [info     ] supervisor_analyzing           doc_scope=system topic='Đạo hàm và ứng dụng' user_id=perf_benchmark_tier1
2026-04-09 13:55:41 [info     ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3-flash-preview temperature=0
2026-04-09 13:55:41 [info     ] rate_limiter_init_bucket       capacity=5.0 refill_rate=1.0 rpm=60
2026-04-09 13:55:41 [info     ] rate_limiter_init_semaphore    max_concurrent=10
2026-04-09 13:55:43 [warning  ] unsupported_content_type_defaulting_to_math detected=
2026-04-09 13:55:43 [info     ] supervisor_classified          content_type=math
2026-04-09 13:55:43 [info     ] routing_to_agent               conte

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-04-09 13:55:46 [debug    ] retrieved_documents            count=12
2026-04-09 13:55:46 [info     ] context_retrieved              chunks=12 sources=12
2026-04-09 13:55:46 [info     ] math_agent_context_retrieved   context_chunks=12
2026-04-09 13:55:46 [info     ] creating_generation_llm        location=global max_output_tokens=8192 model=gemini-3-flash-preview temperature=0.7
2026-04-09 13:56:09 [info     ] math_agent_code_execution_done code_traces_count=2 generation_attempt=1 text_length=4974
2026-04-09 13:56:09 [info     ] creating_generation_llm        location=global max_output_tokens=8192 model=gemini-3-flash-preview temperature=0.3


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-04-09 13:56:26 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=6
2026-04-09 13:56:27 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-04-09 13:56:27 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=13
2026-04-09 13:56:27 [info     ] math_agent_generated           items_count=13
2026-04-09 13:56:27 [info     ] reviewer_reviewing             items_count=13 iteration=0
2026-04-09 13:56:27 [info     ] creating_review_llm            location=global max_output_tokens=4096 model=gemini-3-flash-preview temperature=0.0
2026-04-09 13:56:49 [error    ] reviewer_failed                error='Failed to parse ReviewBatch from completion {"reviews": [{"item_index": 0, "passed": true, "score": 1.0, "feedback": "Chính xác, rõ ràng, đúng chuẩn kiến thức nhận biết."}, {"item_index": 1, "passed": true, "score": 1.0, "feedback": "Chính xác, nội dung phù hợp với chương trình đạo hàm lớp 11/12."}, {"item_index": 2, "passed": true, "score": 

...

2026-04-09 14:02:48 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-04-09 14:02:58 [warning  ] math_agent_batch_parse_exception attempt=1 batch=1 error='Invalid json output: \nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE '
2026-04-09 14:04:04 [info     ] math_agent_batch_parsed        attempt=2 batch=1 items=7
2026-04-09 14:04:04 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=14
2026-04-09 14:04:04 [info     ] math_agent_generated           items_count=14
2026-04-09 14:04:04 [info     ] reviewer_reviewing             items_count=14 iteration=0
2026-04-09 14:04:23 [error    ] reviewer_failed                error='Failed to parse ReviewBatch from completion {"reviews": [{"item_index": 0, "passed": true, "score": 1.0, "feedback": "Accuracy is high; calculation of derivative and roots is correct. Vietnamese terminology is standard."}, {"item_index": 1, "passed": true, "score": 1.0,

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-04-09 14:05:32 [debug    ] retrieved_documents            count=30
2026-04-09 14:05:32 [info     ] context_retrieved              chunks=30 sources=30
2026-04-09 14:05:32 [info     ] math_agent_context_retrieved   context_chunks=30
2026-04-09 14:06:34 [info     ] math_agent_code_execution_done code_traces_count=4 generation_attempt=1 text_length=13474


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-04-09 14:06:59 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=7
2026-04-09 14:07:03 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-04-09 14:07:28 [info     ] math_agent_batch_parsed        attempt=1 batch=4 items=4
2026-04-09 14:08:16 [warning  ] math_agent_batch_parse_exception attempt=1 batch=2 error='Invalid json output: \nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE '
2026-04-09 14:08:36 [info     ] math_agent_batch_parsed        attempt=2 batch=2 items=7
2026-04-09 14:08:36 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=25
2026-04-09 14:08:36 [info     ] math_agent_generated           items_count=25
2026-04-09 14:08:36 [info     ] reviewer_reviewing             items_count=25 iteration=0
2026-04-09 14:09:02 [error    ] reviewer_failed                error='Invalid json output: \nFor troubleshooting, visit: https://docs.langchain.com/oss/python/lan

## Section 4: Verdict & Comparison

In [4]:
print("\n" + "=" * 80)
print("TIER 1 PERFORMANCE BENCHMARK — RESULTS")
print("=" * 80)
print(f"{'Difficulty':<20} {'Delivered':>9} {'Rate':>7} {'Baseline':>9} {'Now':>8} {'Speedup':>9} {'Status':>8}")
print("-" * 80)

all_passed = True
for diff, r in e2e_results.items():
    status = "✅ PASS" if r["passed"] else "❌ FAIL"
    if not r["passed"]:
        all_passed = False
    print(
        f"{diff:<20} {r['delivered']:>5}/{r['requested']:<3} {r['pct']:>6.0f}% "
        f"{r['baseline']:>8.1f}s {r['elapsed']:>7.1f}s {r['speedup_pct']:>+8.1f}% {status:>8}"
    )

print("-" * 80)
total_d = sum(r["delivered"] for r in e2e_results.values())
total_r = sum(r["requested"] for r in e2e_results.values())
total_pct = total_d / total_r * 100 if total_r else 0
total_time = sum(r["elapsed"] for r in e2e_results.values())
total_speedup = (BASELINE_TOTAL - total_time) / BASELINE_TOTAL * 100
print(
    f"{'TOTAL':<20} {total_d:>5}/{total_r:<3} {total_pct:>6.0f}% "
    f"{BASELINE_TOTAL:>8.1f}s {total_time:>7.1f}s {total_speedup:>+8.1f}%"
)

print("\n" + "=" * 80)
print("PERFORMANCE TARGETS")
print("=" * 80)

# Target 1: 100% delivery maintained
delivery_ok = all_passed
print(f"  {'✅' if delivery_ok else '❌'} 100% delivery: {total_d}/{total_r} ({total_pct:.0f}%)")

# Target 2: high_application <= 800s
ha_time = e2e_results["high_application"]["elapsed"]
ha_ok = ha_time <= 800
print(f"  {'✅' if ha_ok else '⚠️'} high_application ≤ 800s: {ha_time:.1f}s")

# Target 3: total <= 1400s
total_ok = total_time <= 1400
print(f"  {'✅' if total_ok else '⚠️'} Total ≤ 1400s: {total_time:.1f}s")

print("\n" + "=" * 80)
if delivery_ok:
    print(f"🎯 100% delivery maintained | Total speedup: {total_speedup:+.1f}% ({BASELINE_TOTAL:.0f}s → {total_time:.0f}s)")
else:
    failed = [d for d, r in e2e_results.items() if not r["passed"]]
    print(f"❌ DELIVERY REGRESSION — Failed: {', '.join(failed)}")
print("=" * 80)


TIER 1 PERFORMANCE BENCHMARK — RESULTS
Difficulty           Delivered    Rate  Baseline      Now   Speedup   Status
--------------------------------------------------------------------------------
recall                  10/10     100%    125.1s   105.2s    +15.9%   ✅ PASS
comprehension           10/10     100%    122.7s   226.4s    -84.5%   ✅ PASS
application             10/10     100%    539.9s   255.8s    +52.6%   ✅ PASS
high_application        10/10     100%   1290.8s   255.1s    +80.2%   ✅ PASS
--------------------------------------------------------------------------------
TOTAL                   40/40     100%   2078.5s   842.4s    +59.5%

PERFORMANCE TARGETS
  ✅ 100% delivery: 40/40 (100%)
  ✅ high_application ≤ 800s: 255.1s
  ✅ Total ≤ 1400s: 842.4s

🎯 100% delivery maintained | Total speedup: +59.5% (2078s → 842s)
